In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine

In [4]:
#Define file path and load raw data
BASE_DIR = os.path.dirname(os.getcwd())
CSV_PATH = os.path.join(BASE_DIR,'data','Churn_Modelling.csv')
df = pd.read_csv(CSV_PATH)

In [5]:
# Output the column names of the Pandas dataframe
df.columns

Index(['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography',
       'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Exited'],
      dtype='str')

In [6]:
#Clean and standardize columns
df_clean = df.drop(columns=['RowNumber']).copy()
df_clean.rename(
    columns={
        "CustomerId": "customer_id",
        "Surname": "surname",
        "CreditScore": "credit_score",
        "Geography": "geography",
        "Gender": "gender",
        "Age": "age",
        "Tenure": "tenure",
        "Balance": "balance",
        "NumOfProducts": "num_of_products",
        "HasCrCard": "has_cr_card",
        "IsActiveMember": "is_active_member",
        "EstimatedSalary": "estimated_salary",
        "Exited": "exited",
    },
    inplace=True
)

In [8]:
# Output the updated column names of the Pandas dataframe after transformations
df_clean.columns

Index(['customer_id', 'surname', 'credit_score', 'geography', 'gender', 'age',
       'tenure', 'balance', 'num_of_products', 'has_cr_card',
       'is_active_member', 'estimated_salary', 'exited'],
      dtype='str')

In [10]:
#Precision rounding for currency fields
df_clean['balance'] = df_clean['balance'].round(2)
df_clean['estimated_salary'] = df_clean['estimated_salary'].round(2)

In [11]:
#Feature engineering for SQL & Streamlit reporting
df_clean['churn_status'] = df_clean['exited'].map({1:'Churned',0:'Retained'})
df_clean['has_credit_card_label'] = df_clean['has_cr_card'].map({1:'Yes',0:'No'})
df_clean['is_active_member_label'] = df_clean['is_active_member'].map(
    {1:'Active',0:'Inactive'}
)

In [13]:
#Categorical Bins
age_bins = [0,29,39,49,59,100]
age_labels = ['< 30','30-39','40-49','50-59','60+']
df_clean['age_group'] = pd.cut(
    df_clean['age'], bins=age_bins, labels=age_labels
).astype(str)

credit_bins = [0, 579, 669, 739, 799, 900]
credit_labels = [
    "Poor (< 580)",
    "Fair (580-669)",
    "Good (670-739)",
    "Very Good (740-799)",
    "Excellent (800+)",
]
df_clean['credit_tier'] = pd.cut(
    df_clean['credit_score'], bins=credit_bins, labels=credit_labels
).astype(str)

In [15]:
#Database Connection (SQL Server)
SERVER='localhost'
DATABASE='BankChurnDB'
DRIVER = 'ODBC Driver 17 for SQL Server'

connection_string = (
    f'mssql+pyodbc://{SERVER}/{DATABASE}?driver={DRIVER}&trusted_connection=yes'
)
engine = create_engine(connection_string)

In [16]:
#Push cleaned DataFrame to SQL Server
df_clean.to_sql('bank_churn',con=engine,if_exists='replace',index=False)
print(
    f"Successfully ingested {len(df_clean)} cleaned records into '{DATABASE}.dbo.bank_churn'."
)

C:\Users\Hassa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pandas\io\sql.py:1649: SAWarning: Unrecognized server version info '17.0.1135.8'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


Successfully ingested 10000 cleaned records into 'BankChurnDB.dbo.bank_churn'.
